<a href="https://colab.research.google.com/github/LozaMengistu/Credit-Card-Fraud-Detection-/blob/main/Agentic_AI_System_for_Credit_Card_Fraud_Detection_and_Time_Based_Risk_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile("creditcard.csv (1).zip", 'r') as zip_ref:
    zip_ref.extractall()

print("Unzipped successfully")

In [ ]:
import os
print(os.listdir())

In [ ]:
import pandas as pd

df = pd.read_csv("creditcard.csv")

print(df.shape)
print(df.head())

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model trained")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:\n")
print(classification_report(y_test, y_pred, digits=4))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
import matplotlib.pyplot as plt

df["hour"] = (df["Time"] // 3600).astype(int)

fraud_by_hour = df.groupby("hour")["Class"].sum()

plt.figure(figsize=(10,5))
plt.plot(fraud_by_hour.index, fraud_by_hour.values, marker='o')
plt.title("Fraud Transactions Over Time")
plt.xlabel("Hour")
plt.ylabel("Fraud Count")
plt.show()

In [ ]:
def detection_agent(transaction, model):
    prob = model.predict_proba(transaction)[0][1]
    pred = int(prob >= 0.5)
    return {"fraud": pred, "prob": round(prob, 4)}

def pattern_agent(transaction):
    alerts = []

    amount = float(transaction["Amount"].iloc[0])
    time_val = float(transaction["Time"].iloc[0])

    if amount > 1000:
        alerts.append("High transaction amount")

    hour = int((time_val // 3600) % 24)
    if hour < 5:
        alerts.append("Unusual transaction time")

    return alerts

def explanation_agent(detection, patterns):
    if detection["fraud"] == 1:
        reason = ", ".join(patterns) if patterns else "Suspicious pattern"
        return f"Fraud detected (prob={detection['prob']}). Reason: {reason}"
    else:
        return f"Transaction is normal (prob={detection['prob']})"

In [ ]:
sample = X_test.iloc[[0]].copy()

d = detection_agent(sample, model)
p = pattern_agent(sample)
e = explanation_agent(d, p)

print("Detection:", d)
print("Pattern:", p)
print("Explanation:", e)

In [ ]:
def detection_agent(transaction, model):
    prob = float(model.predict_proba(transaction)[0][1])
    pred = int(prob >= 0.5)
    return {"fraud": pred, "prob": round(prob, 4)}

In [ ]:
def detection_agent(transaction, model):
    prob = float(model.predict_proba(transaction)[0][1])
    pred = int(prob >= 0.5)
    return {"fraud": pred, "prob": round(prob, 4)}

def pattern_agent(transaction):
    alerts = []

    amount = float(transaction["Amount"].iloc[0])
    time_val = float(transaction["Time"].iloc[0])

    if amount > 1000:
        alerts.append("High transaction amount")

    hour = int((time_val // 3600) % 24)
    if hour < 5:
        alerts.append("Unusual transaction time")

    return alerts

def explanation_agent(detection, patterns):
    if detection["fraud"] == 1:
        reason = ", ".join(patterns) if patterns else "Suspicious pattern"
        return f"Fraud detected (prob={detection['prob']}). Reason: {reason}"
    else:
        return f"Transaction is normal (prob={detection['prob']})"

In [ ]:
sample = X_test.iloc[[0]].copy()

d = detection_agent(sample, model)
p = pattern_agent(sample)
e = explanation_agent(d, p)

print("Detection:", d)
print("Pattern:", p)
print("Explanation:", e)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train, y_train)
lr_prob = lr.predict_proba(X_test)[:, 1]

# Decision Tree
dt = DecisionTreeClassifier(class_weight="balanced", random_state=42)
dt.fit(X_train, y_train)
dt_prob = dt.predict_proba(X_test)[:, 1]

# Random Forest (already trained)
rf_prob = model.predict_proba(X_test)[:, 1]

print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, lr_prob))
print("Decision Tree ROC-AUC:", roc_auc_score(y_test, dt_prob))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_prob))

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="Class", y="Amount", data=df)
plt.title("Transaction Amount by Class")
plt.show()

In [ ]:
sns.boxplot(x="Class", y="Time", data=df)
plt.title("Transaction Time by Class")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, rf_prob)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label="Random Forest (AUC=0.95)")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
# -----------------------------
# Agent 1: Detection Agent
# -----------------------------
def detection_agent(transaction, model):
    """
    Uses the trained ML model to predict whether a transaction is fraud.
    """
    prob = float(model.predict_proba(transaction)[0][1])
    pred = int(prob >= 0.5)

    return {
        "fraud": pred,
        "probability": round(prob, 4)
    }


# -----------------------------
# Agent 2: Pattern Agent
# -----------------------------
def pattern_agent(transaction):
    """
    Looks for simple suspicious patterns in the transaction.
    """
    alerts = []

    amount = float(transaction["Amount"].iloc[0])
    time_val = float(transaction["Time"].iloc[0])

    if amount > 1000:
        alerts.append("High transaction amount")

    hour = int((time_val // 3600) % 24)
    if hour < 5:
        alerts.append("Unusual transaction time")

    return alerts


# -----------------------------
# Agent 3: Explanation Agent
# -----------------------------
def explanation_agent(detection_result, pattern_alerts):
    """
    Combines the model prediction and pattern analysis
    into a human-readable explanation.
    """
    if detection_result["fraud"] == 1:
        if pattern_alerts:
            reason = ", ".join(pattern_alerts)
        else:
            reason = "Model detected suspicious transaction behavior"

        return (
            f"Fraud detected. "
            f"Probability = {detection_result['probability']}. "
            f"Reason: {reason}."
        )
    else:
        return (
            f"Transaction is normal. "
            f"Fraud probability = {detection_result['probability']}."
        )

In [ ]:
# Test on one sample transaction
sample = X_test.iloc[[0]].copy()

detect_result = detection_agent(sample, model)
pattern_result = pattern_agent(sample)
explanation_result = explanation_agent(detect_result, pattern_result)

print("Detection Agent Output:", detect_result)
print("Pattern Agent Output:", pattern_result)
print("Explanation Agent Output:", explanation_result)

In [ ]:
# Test on a known fraud sample
fraud_samples = X_test[y_test == 1]
sample_fraud = fraud_samples.iloc[[0]].copy()

detect_result_fraud = detection_agent(sample_fraud, model)
pattern_result_fraud = pattern_agent(sample_fraud)
explanation_result_fraud = explanation_agent(detect_result_fraud, pattern_result_fraud)

print("Detection Agent Output:", detect_result_fraud)
print("Pattern Agent Output:", pattern_result_fraud)
print("Explanation Agent Output:", explanation_result_fraud)

In [ ]:
def explanation_agent(detection_result, pattern_alerts):
    status = "FRAUD" if detection_result["fraud"] == 1 else "NORMAL"
    prob = detection_result["probability"]

    report = "\n🚨 FRAUD ANALYSIS REPORT\n"
    report += f"\nPrediction: {status}"
    report += f"\nConfidence Score: {prob}"

    # Pattern section
    report += "\n\n🔍 Pattern Analysis:\n"
    if pattern_alerts:
        for alert in pattern_alerts:
            report += f"- {alert}\n"
    else:
        report += "- No obvious rule-based patterns detected\n"

    # AI explanation
    report += "\n🤖 AI Insight:\n"
    if detection_result["fraud"] == 1:
        report += "The model detected unusual feature behavior compared to normal transactions.\n"
    else:
        report += "The transaction aligns with normal behavioral patterns.\n"

    # Final decision
    report += "\n📊 Final Decision:\n"
    if detection_result["fraud"] == 1:
        report += "This transaction is highly likely to be fraudulent."
    else:
        report += "This transaction is likely legitimate."

    return report

In [ ]:
def pattern_agent(transaction):
    alerts = []

    amount = float(transaction["Amount"].iloc[0])
    time_val = float(transaction["Time"].iloc[0])

    hour = int((time_val // 3600) % 24)

    if amount > 1000:
        alerts.append("High transaction amount")

    if amount < 1:
        alerts.append("Very small suspicious amount")

    if hour < 5:
        alerts.append("Transaction occurred during unusual hours (midnight–early morning)")

    return alerts

In [ ]:
sample = X_test.iloc[[0]].copy()

d = detection_agent(sample, model)
p = pattern_agent(sample)
e = explanation_agent(d, p)

print(e)

In [ ]:
def detection_agent(transaction, model):
    prob = float(model.predict_proba(transaction)[0][1])
    pred = int(prob >= 0.5)

    if prob >= 0.8:
        risk_level = "High"
    elif prob >= 0.4:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    return {
        "fraud": pred,
        "probability": round(prob, 4),
        "risk_level": risk_level
    }

In [ ]:
def pattern_agent(transaction):
    alerts = []

    amount = float(transaction["Amount"].iloc[0])
    time_val = float(transaction["Time"].iloc[0])

    hour = int((time_val // 3600) % 24)

    if amount > 1000:
        alerts.append("High transaction amount")

    if amount < 1:
        alerts.append("Very small suspicious amount")

    if hour < 5:
        alerts.append("Transaction occurred during unusual hours (midnight–early morning)")

    return alerts

In [ ]:
sample = X_test.iloc[[0]].copy()

d = detection_agent(sample, model)
p = pattern_agent(sample)
e = explanation_agent(d, p)

print(e)

In [ ]:
fraud_samples = X_test[y_test == 1]
sample_fraud = fraud_samples.iloc[[0]].copy()

d = detection_agent(sample_fraud, model)
p = pattern_agent(sample_fraud)
e = explanation_agent(d, p)

print(e)

In [ ]:
fraud_docs = [
    {
        "title": "High-Value Transaction Review",
        "text": """
High-value transactions should be reviewed carefully, especially when they are unusual
relative to typical customer spending patterns. Analysts should compare the transaction
amount to normal historical behavior and check for additional anomaly signals.
"""
    },
    {
        "title": "Off-Hours Transaction Review",
        "text": """
Transactions occurring during unusual hours, such as late night or early morning,
may indicate suspicious behavior. These cases should be reviewed together with
amount, transaction velocity, and other anomaly indicators.
"""
    },
    {
        "title": "Behavioral Anomaly Review",
        "text": """
When multiple behavioral features deviate strongly from normal patterns, the case
may represent a fraud scenario. Strong anomaly patterns with high model confidence
should be escalated for manual review.
"""
    },
    {
        "title": "False Positive Review Guidance",
        "text": """
Not all flagged transactions are fraudulent. False positives may occur when a legitimate
customer makes an unusual purchase. Analysts should review transaction context before
final escalation.
"""
    },
    {
        "title": "Fraud Escalation Policy",
        "text": """
Transactions with high fraud probability and multiple suspicious indicators should be
flagged for manual review. If the transaction shows both anomaly patterns and high-risk
signals, escalation is recommended.
"""
    }
]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

doc_texts = [doc["text"] for doc in fraud_docs]
doc_titles = [doc["title"] for doc in fraud_docs]

vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(doc_texts)

In [ ]:
def rag_retrieve(query, top_k=2):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, doc_vectors).flatten()

    top_indices = scores.argsort()[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "title": doc_titles[idx],
            "text": doc_texts[idx].strip(),
            "score": round(float(scores[idx]), 4)
        })

    return results

In [ ]:
test_results = rag_retrieve("What should be reviewed in a high-risk fraud case?")
print(test_results)

In [ ]:
print(df.isnull().sum().sum())
print(df.isnull().sum())

In [ ]:
# -----------------------------
# Missing-value check
# -----------------------------
print("Total missing values in dataset:", df.isnull().sum().sum())
print("\nMissing values by column:")
print(df.isnull().sum())

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Random Forest trained with class_weight='balanced'")

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

lr_model.fit(X_train, y_train)

print("Logistic Regression trained with class_weight='balanced'")

In [ ]:
import numpy as np

locations = ["USA", "UK", "Canada", "Germany", "India", "Brazil"]

np.random.seed(42)
df["location"] = np.random.choice(locations, size=len(df))

df[["location"]].head()

In [ ]:
df["home_location"] = "USA"

In [ ]:
df["location_mismatch"] = df["location"] != df["home_location"]

df[["location", "home_location", "location_mismatch"]].head()

In [ ]:
def pattern_agent(transaction, df_reference):
    alerts = []
    details = {}

    amount = float(transaction["Amount"].iloc[0])
    time_val = float(transaction["Time"].iloc[0])
    hour = int((time_val // 3600) % 24)

    location = transaction["location"].iloc[0]
    home_location = transaction["home_location"].iloc[0]

    # Amount
    if amount > df_reference["Amount"].quantile(0.99):
        alerts.append("Extremely high transaction amount")
        details["amount"] = "above 99th percentile"

    # Time
    if hour < 5:
        alerts.append("Unusual transaction time")
        details["time"] = f"hour={hour}"

    # Location anomaly
    if location != home_location:
        alerts.append("Transaction from new/unusual location")
        details["location"] = f"{location} vs home={home_location}"

    return alerts, details

In [ ]:
def scenario_agent(detection_result, pattern_alerts):
    if detection_result["fraud"] == 0:
        return "Legitimate transaction pattern"

    if "Transaction from new/unusual location" in pattern_alerts:
        return "Possible location-based fraud or account compromise"

    if "Extremely high transaction amount" in pattern_alerts:
        return "High-value fraud scenario"

    if "Unusual transaction time" in pattern_alerts:
        return "Off-hours fraud scenario"

    return "General fraud risk scenario"

In [ ]:
# =========================================================
# 1. ADD SIMULATED LOCATION FEATURES
# =========================================================
import numpy as np
import pandas as pd

np.random.seed(42)

locations = ["USA", "UK", "Canada", "Germany", "India", "Brazil"]
df["location"] = np.random.choice(locations, size=len(df))
df["home_location"] = "USA"
df["location_mismatch"] = (df["location"] != df["home_location"]).astype(int)

# Rebuild X/y and split again so the new columns are included
from sklearn.model_selection import train_test_split

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Updated train shape:", X_train.shape)
print("Updated test shape:", X_test.shape)

In [ ]:
# =========================================================
# 2. RETRAIN MODEL WITH NEW FEATURES
# =========================================================
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

# Convert location strings to numeric with one-hot encoding
X_train_model = pd.get_dummies(X_train, columns=["location", "home_location"], drop_first=False)
X_test_model = pd.get_dummies(X_test, columns=["location", "home_location"], drop_first=False)

# Make sure test columns match train columns
X_test_model = X_test_model.reindex(columns=X_train_model.columns, fill_value=0)

model.fit(X_train_model, y_train)
print("Model retrained with simulated location features.")

In [ ]:
# =========================================================
# 3. AGENTIC AI SYSTEM
# =========================================================
def detection_agent(transaction_model_ready, model):
    prob = float(model.predict_proba(transaction_model_ready)[0][1])
    pred = int(prob >= 0.5)

    if prob >= 0.90:
        risk_level = "Critical"
    elif prob >= 0.70:
        risk_level = "High"
    elif prob >= 0.40:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    return {
        "fraud": pred,
        "probability": round(prob, 4),
        "risk_level": risk_level
    }


def pattern_agent(transaction_raw, df_reference):
    alerts = []
    details = {}

    amount = float(transaction_raw["Amount"].iloc[0])
    time_val = float(transaction_raw["Time"].iloc[0])
    hour = int((time_val // 3600) % 24)

    location = transaction_raw["location"].iloc[0]
    home_location = transaction_raw["home_location"].iloc[0]

    q95 = df_reference["Amount"].quantile(0.95)
    q99 = df_reference["Amount"].quantile(0.99)

    if amount > q99:
        alerts.append("Extremely high transaction amount")
        details["amount_context"] = "above 99th percentile"
    elif q95 <= amount < q99:
        alerts.append("High transaction amount")
        details["amount_context"] = "between 95th and 99th percentile"

    if amount < 1:
        alerts.append("Very small suspicious amount")
        details["small_amount_flag"] = "below 1 unit"

    if hour < 5:
        alerts.append("Transaction occurred during unusual early-morning hours")
        details["time_context"] = f"hour={hour}"

    if location != home_location:
        alerts.append("Transaction from new/unusual location")
        details["location_context"] = f"{location} vs home={home_location}"

    # simple anomaly count using V-features
    feature_cols = [c for c in transaction_raw.columns if c.startswith("V")]
    anomaly_count = 0
    flagged_features = []

    for col in feature_cols:
        mean_val = df_reference[col].mean()
        std_val = df_reference[col].std()
        if std_val > 0:
            z = abs((float(transaction_raw[col].iloc[0]) - mean_val) / std_val)
            if z > 3:
                anomaly_count += 1
                flagged_features.append(col)

    if anomaly_count >= 3:
        alerts.append("Multiple anonymized behavioral features show abnormal deviation")
        details["feature_context"] = flagged_features[:5]

    return alerts, details


def scenario_agent(detection_result, pattern_alerts):
    if detection_result["fraud"] == 0:
        return "Likely legitimate transaction pattern"

    if (
        "Transaction from new/unusual location" in pattern_alerts
        and "Extremely high transaction amount" in pattern_alerts
    ):
        return "Possible account compromise or location-based fraud"

    if "Transaction from new/unusual location" in pattern_alerts:
        return "Possible location-based fraud scenario"

    if "Multiple anonymized behavioral features show abnormal deviation" in pattern_alerts:
        return "Behavioral anomaly fraud scenario"

    if "Transaction occurred during unusual early-morning hours" in pattern_alerts:
        return "Off-hours fraud risk scenario"

    if "Extremely high transaction amount" in pattern_alerts or "High transaction amount" in pattern_alerts:
        return "High-value transaction anomaly scenario"

    return "General suspicious transaction scenario"

In [ ]:
# =========================================================
# 4. SIMPLE RAG KNOWLEDGE BASE
# =========================================================
fraud_docs = [
    {
        "title": "High-Value Transaction Review",
        "text": """
High-value transactions should be reviewed carefully, especially when they are unusual
relative to typical customer spending patterns. Analysts should compare the transaction
amount to expected behavior and review other anomaly indicators.
"""
    },
    {
        "title": "Off-Hours Fraud Review",
        "text": """
Transactions occurring during unusual hours, such as late night or early morning,
may indicate suspicious behavior. These should be reviewed with amount and behavioral context.
"""
    },
    {
        "title": "Location-Based Fraud Guidance",
        "text": """
Transactions from new or unusual locations may suggest account compromise, travel-related anomalies,
or unauthorized card use. These alerts should be verified before escalation.
"""
    },
    {
        "title": "Behavioral Anomaly Review",
        "text": """
When multiple anonymized behavioral features deviate from normal patterns, the transaction may represent
a fraud scenario. High-confidence anomaly cases should be escalated for manual review.
"""
    },
    {
        "title": "False Positive Review Guidance",
        "text": """
Not all flagged transactions are fraudulent. False positives may occur when legitimate customers make
unusual purchases or travel. Review transaction context before final escalation.
"""
    },
    {
        "title": "Fraud Escalation Policy",
        "text": """
Transactions with high fraud probability and multiple suspicious indicators should be flagged for review.
If both anomaly patterns and high-risk contextual signals are present, escalation is recommended.
"""
    }
]

In [ ]:
# =========================================================
# 5. BUILD RAG RETRIEVER
# =========================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

doc_texts = [doc["text"] for doc in fraud_docs]
doc_titles = [doc["title"] for doc in fraud_docs]

vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(doc_texts)

def rag_retrieve(query, top_k=2):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, doc_vectors).flatten()
    top_idx = scores.argsort()[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            "title": doc_titles[idx],
            "text": doc_texts[idx].strip(),
            "score": round(float(scores[idx]), 4)
        })
    return results


def rag_verification_agent(detection_result, scenario_label, pattern_alerts):
    query = f"""
    prediction={detection_result['fraud']},
    risk={detection_result['risk_level']},
    scenario={scenario_label},
    alerts={'; '.join(pattern_alerts) if pattern_alerts else 'none'}.
    What review guidance applies, what should an analyst check next,
    and could this be a false positive?
    """
    return rag_retrieve(query, top_k=2)

In [ ]:
# =========================================================
# 6. FINAL OUTCOME REPORT
# =========================================================
def format_rag_output(rag_results):
    out = "\n[RAG Verification]"
    for i, r in enumerate(rag_results, start=1):
        out += f"\n\nSource {i}: {r['title']} (score={r['score']})"
        out += f"\n{r['text']}"
    return out


def final_report_agent(detection_result, pattern_alerts, pattern_details, scenario_label, rag_results):
    status = "FRAUD" if detection_result["fraud"] == 1 else "NORMAL"
    prob = detection_result["probability"]
    risk = detection_result["risk_level"]

    report = "\n========== TRANSACTION RISK ANALYSIS REPORT ==========\n"
    report += f"\nPrediction: {status}"
    report += f"\nFraud Probability: {prob}"
    report += f"\nRisk Level: {risk}"
    report += f"\nScenario Label: {scenario_label}"

    report += "\n\n[Pattern Analysis]"
    if pattern_alerts:
        for alert in pattern_alerts:
            report += f"\n- {alert}"
    else:
        report += "\n- No major rule-based alert was triggered"

    report += "\n\n[Supporting Details]"
    if pattern_details:
        for k, v in pattern_details.items():
            report += f"\n- {k}: {v}"
    else:
        report += "\n- No additional supporting details"

    report += "\n\n[AI Insight]"
    if detection_result["fraud"] == 1:
        report += (
            "\nThe model flagged this transaction because its overall pattern differs "
            "from normal transactions. The decision combines model confidence, timing, "
            "amount context, location mismatch, and anonymized feature deviation."
        )
    else:
        report += (
            "\nThe model found this transaction consistent with legitimate activity. "
            "No major anomaly signals were triggered."
        )

    report += format_rag_output(rag_results)

    report += "\n\n[Final Decision]"
    if detection_result["fraud"] == 1:
        report += "\nThis transaction should be flagged for manual review as a likely fraud case."
    else:
        report += "\nThis transaction appears legitimate and does not require escalation."

    return report

In [ ]:
# =========================================================
# 7. HELPER TO PREPARE A SINGLE ROW FOR THE MODEL
# =========================================================
def prepare_single_transaction_for_model(transaction_raw, train_columns):
    tx_model = pd.get_dummies(transaction_raw, columns=["location", "home_location"], drop_first=False)
    tx_model = tx_model.reindex(columns=train_columns, fill_value=0)
    return tx_model

In [ ]:
# =========================================================
# 8. DEMO: FRAUD EXAMPLE OUTCOME
# =========================================================
fraud_samples = X_test[y_test == 1]
sample_fraud_raw = fraud_samples.iloc[[0]].copy()

sample_fraud_model = prepare_single_transaction_for_model(sample_fraud_raw, X_train_model.columns)

d = detection_agent(sample_fraud_model, model)
p_alerts, p_details = pattern_agent(sample_fraud_raw, X_train)
s = scenario_agent(d, p_alerts)
rag_results = rag_verification_agent(d, s, p_alerts)

fraud_report = final_report_agent(d, p_alerts, p_details, s, rag_results)
print(fraud_report)

In [ ]:
# =========================================================
# 9. DEMO: NORMAL EXAMPLE OUTCOME
# =========================================================
normal_samples = X_test[y_test == 0]
sample_normal_raw = normal_samples.iloc[[0]].copy()

sample_normal_model = prepare_single_transaction_for_model(sample_normal_raw, X_train_model.columns)

d = detection_agent(sample_normal_model, model)
p_alerts, p_details = pattern_agent(sample_normal_raw, X_train)
s = scenario_agent(d, p_alerts)
rag_results = rag_verification_agent(d, s, p_alerts)

normal_report = final_report_agent(d, p_alerts, p_details, s, rag_results)
print(normal_report)